# DriveMind AI — Intent Classification (TF-IDF + Logistic Regression)

Trains and evaluates the baseline classifier directly in-notebook (same code path as `scripts/train_baseline.py`).
All metrics below are computed from an actual `fit`/`predict` run against the real dataset.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import seaborn as sns

from src.config.settings import load_yaml_config
from src.evaluation.metrics import compute_classification_metrics
from src.evaluation.model_evaluation import get_split, load_annotated_dataframe
from src.models.baseline import BaselineIntentClassifier

cfg = load_yaml_config('config.yaml')
df = load_annotated_dataframe(Path.cwd().parent / cfg['dataset']['annotated_path'])
train_df, val_df, test_df = get_split(df, 'train'), get_split(df, 'val'), get_split(df, 'test')
len(train_df), len(val_df), len(test_df)

In [ ]:
model_cfg = cfg['baseline_model']
model = BaselineIntentClassifier(
    max_features=model_cfg['tfidf_max_features'],
    ngram_range=tuple(model_cfg['tfidf_ngram_range']),
    C=model_cfg['logreg_C'],
    max_iter=model_cfg['logreg_max_iter'],
)
model.fit(train_df['text'].tolist(), train_df['intent'].tolist())
print('Model trained.')

## Test set evaluation

In [ ]:
all_labels = sorted(df['intent'].unique().tolist())
preds = model.predict(test_df['text'].tolist())
metrics = compute_classification_metrics(test_df['intent'].tolist(), preds, all_labels)
print(f"Accuracy: {metrics['accuracy']}")
print(f"Macro F1: {metrics['f1_macro']}")
print(f"Weighted F1: {metrics['f1_weighted']}")

## Confusion matrix

In [ ]:
import numpy as np
cm = np.array(metrics['confusion_matrix'])
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, xticklabels=all_labels, yticklabels=all_labels, cmap='Blues', ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Try a few example utterances interactively

In [ ]:
examples = [
    'Make it warmer',
    'Find a charging station near Berlin with at least 150 kW',
    'Play some jazz',
    'What time is it',
]
for ex in examples:
    intent, confidence = model.predict_one(ex)
    print(f'{ex!r} -> {intent} (confidence={confidence})')